In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/abdulsalamramatu/complete-math-features/new_math_df.csv


In [3]:
import numpy as np
import pandas as pd

In [17]:
math_df=pd.read_csv("/kaggle/input/datasets/abdulsalamramatu/complete-math-features/new_math_df.csv")
mtld_na=math_df.loc[math_df.mtld.isna()].conversation_id.tolist()
math_df=math_df[~math_df['conversation_id'].isin(mtld_na)]

math_df["mtld_log"] = np.log1p(math_df["mtld"])
math_df["word_count_log"] = np.log1p(math_df["word_count"])

In [20]:
def sample_score_bins(df, score, n_total=100, n_bins=4):
    bin_edges=np.percentile(df[score], np.linspace(0,100, n_bins+1))
    bin_edges[0]=-np.inf
    bin_edges[-1]=np.inf
    df['score_bin']=pd.cut(
        df[score], 
        bins=bin_edges, 
        labels=[f'Bin{i+1}' for i  in range(n_bins)],
        include_lowest=True)
    
    samples_per_bin=n_total// n_bins
    reminder=n_total % n_bins
    
    sampled_dfs=[]
    
    for i, bin_label in enumerate(df['score_bin'].cat.categories):
        
        bin_df=df[df['score_bin']==bin_label]
        n_samples=samples_per_bin +(1 if i < reminder else 0)
        n_samples=min(n_samples, len(bin_df))
        
        if n_samples>0:
            sampled_dfs.append(bin_df.sample(n=n_samples, random_state=42))

    sampled_df = pd.concat(sampled_dfs)
    cols_to_keep=['tutor', 'conversation_id', 'student_mistake', 'conversation_id', 'tutors_response']

    columns_to_keep = list(set(cols_to_keep ))
    sampled_df = sampled_df[columns_to_keep]
    #sampled_df = sampled_df.drop(columns=['score_bin'])
    
    return sampled_df, bin_edges

In [21]:
features = ['PressReasoning_prob', 'PressAccuracy_prob', 'Uptake_prob', 'politeness_score', 'agency_score']
sample_dict={}
bin_edges_dict={}
for feature in features:
    print(f"Sampling for {feature}....")
    sampled_df, edges= sample_score_bins(math_df, feature,n_total=100, n_bins=4)
    sample_dict[feature]=sampled_df
    bin_edges_dict[feature]=edges   
    sampled_df.to_csv(f"sampled_{feature}.csv", index=False)

Sampling for PressReasoning_prob....
Sampling for PressAccuracy_prob....
Sampling for Uptake_prob....
Sampling for politeness_score....
Sampling for agency_score....
